In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
import zipfile
from lightgbm import LGBMRegressor



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(os.path.join(path, "Q1_data.csv"))


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(8, 5))
sns.histplot(df["Delivery_Time"], bins=30, edgecolor='black')
plt.title("Target Distribution: delivery_time")
plt.xlabel("delivery_time (minutes)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your code here:
if "Order_ID" in df.columns: df = df.drop("Order_ID", axis=1)
df.head()   #see there is no Order_ID anymore

In [ ]:
# Task 2: Write your code here:

#to check what type of misssing values we have
# 2. Do we have missing values?
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df)

print("Missing values before handeling:", df.isnull().sum().sum())  #output = 311

#handeling them:
#simply any numarical null fill with median
#any categorical fill with mode the most repeated
num_cols_all = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols_all = df.select_dtypes(include=["object", "category"]).columns

for col in num_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols_all:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after handeling:", df.isnull().sum().sum())  #output = 311


In [ ]:
# Task 3: Write your code here:

# 4. Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

df = df.drop_duplicates()
after_dupes = df.duplicated().sum()
print(f"after drop: {after_dupes}")


In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)
#Apply feature scaling for all features (Use StandardScaler)
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)


# Task 4: Write your code here:
# Encode categorical variables using One-Hot Encoding

categorical_cols = df.select_dtypes(include=["object", "category"]).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
feature_cols = df.columns.drop("Delivery_Time") #not including the target
df[feature_cols] = scaler.fit_transform(df[feature_cols])


In [ ]:
# Task 6: Write your code here:

# Regression target so not treated as imbalanced like classification

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]

In [ ]:
# Task 2,3,4,5: Write your code here:

num_cols = X.columns.tolist()

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    # Split data
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_val_fold = y.iloc[val_idx]

    # Scale features
    scaler = StandardScaler()
    X_train_final = scaler.fit_transform(X_train_fold)
    X_val_final = scaler.transform(X_val_fold)

    # Train Random Forest model
    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_final, y_train_fold)

    # Predict on validation set
    preds = model.predict(X_val_final)

    # Evaluate using MAE
    mae = mean_absolute_error(y_val_fold, preds)
    mae_scores.append(mae)

    print(f"Fold {fold} MAE: {mae:.4f}")

# Print average MAE across folds
print(f"\nAverage MAE across folds: {np.mean(mae_scores):.4f}")


In [ ]:
# Task 1: Write your code here:

final_model = RandomForestRegressor(n_estimators=300, random_state=42,n_jobs=-1)
final_model.fit(X, y)
importances = final_model.feature_importances_

plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=X.columns)
plt.title("Feature Importance (RandomForest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred = final_model.predict(X)

plt.figure(figsize=(8, 5))
sns.histplot(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:
# Part 5: Bonus - Ensemble

from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores_ensemble = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    # Split data
    X_train_fold = X.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_val_fold = y.iloc[val_idx]

    # Scale features (same scaler for both models)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_val_scaled = scaler.transform(X_val_fold)

    # this is the Random Forest
    rf_model = RandomForestRegressor(
        n_estimators=40,
        random_state=42,
        n_jobs=-1
    )

    # his is the CatBoost
    cb_model = CatBoostRegressor(
        iterations=40,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=False
    )

    # Train both models
    rf_model.fit(X_train_scaled, y_train_fold)
    cb_model.fit(X_train_scaled, y_train_fold)

    # Predict using both models
    rf_preds = rf_model.predict(X_val_scaled)
    cb_preds = cb_model.predict(X_val_scaled)

    # Average predictions
    avg_preds = (rf_preds + cb_preds) / 2

    # Evaluate using MAE
    mae = mean_absolute_error(y_val_fold, avg_preds)
    mae_scores_ensemble.append(mae)

    print(f"Fold {fold} Ensemble MAE: {mae:.4f}")

# Print average ensemble MAE
print(f"\nAverage Ensemble MAE across folds: {np.mean(mae_scores_ensemble):.4f}")

